In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import os
import glob
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

from NNMFit.utilities import load_pickle

### Load Bestfit Histograms (from step0_2_1_reco_space_graphs.ipynb)

In [ ]:
step0_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/philipp/step0_2_1/graphs"

# One example 2D plot: Taupede Length vs Asymmetry
PLOTS = [
    {
        "det_config": "IC86_pass2_SnowStorm_FTP_HESE_Combined",
        "title": "HESE Combined: Taupede Length vs Asymmetry",
        "scan": ("hese_combined_spectrum_bestfit_13year_round3_NoSystematics_len_easym", "MC"),
        "binning": {
            "Taupede_Distance":   np.linspace(0.1360, 3.0133, 11),   # x-axis
            "Taupede_Asymmetry":  np.linspace(-1.001, 1.001, 11),    # y-axis
        },
        "x_label": "Taupede Length",
        "y_label": "Taupede Asymmetry",
        "log_x": True,
        "log_y": False,
    },
]

In [ ]:
scan_names = list(dict.fromkeys(plot["scan"][0] for plot in PLOTS))

histogram_collection = {}
for scan_name in scan_names:
    print(scan_name)
    histogram_collection[scan_name] = {}

    data_hist_path = glob.glob(os.path.join(step0_path, scan_name, "Data_Histogram.pickle"))
    if len(data_hist_path) != 1:
        raise ValueError(f"Expected exactly one Data_Histogram.pickle for {scan_name}, found {len(data_hist_path)}")
    histogram_collection[scan_name]["data"] = load_pickle(data_hist_path[0])

    mc_hist_path = glob.glob(os.path.join(step0_path, scan_name, "MC_Histogram*.pickle"))
    if len(mc_hist_path) < 1:
        raise ValueError(f"Expected at least one MC_Histogram.pickle for {scan_name}, found {len(mc_hist_path)}")
    histogram_collection[scan_name]["mc"] = load_pickle(mc_hist_path[0])

### Plotting Functions

In [ ]:
def get_2d_histogram(histogram_dict, det_config, binning):
    """Reshape the flat histogram array into a 2D grid (x_bins, y_bins)."""
    shape = tuple(edges.shape[0] - 1 for edges in binning.values())
    return np.reshape(histogram_dict["histograms"][det_config], shape)


def plot_2d(ax, h2d, x_edges, y_edges, title, norm=None, cmap="viridis", **pcolor_kw):
    """Plot a 2D histogram with pcolormesh; returns the mappable for colorbar."""
    # h2d shape: (n_x_bins, n_y_bins) — transpose so x is horizontal
    mesh = ax.pcolormesh(x_edges, y_edges, h2d.T, norm=norm, cmap=cmap, **pcolor_kw)
    ax.set_title(title)
    return mesh

### 2D Plots: Data | MC | Data/MC

In [ ]:
save_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/philipp/step0_2_1/plots_2D"
os.makedirs(save_path, exist_ok=True)

for plot in PLOTS:
    det_config          = plot["det_config"]
    scan_name, mc_label = plot["scan"]
    binning             = plot["binning"]
    x_key, y_key        = list(binning.keys())
    x_edges             = binning[x_key]
    y_edges             = binning[y_key]

    data_h2d = get_2d_histogram(histogram_collection[scan_name]["data"], det_config, binning)
    mc_h2d   = get_2d_histogram(histogram_collection[scan_name]["mc"],   det_config, binning)

    # shared colour scale for data and MC
    vmax = max(data_h2d.max(), mc_h2d.max())
    norm = mcolors.Normalize(vmin=0, vmax=vmax)

    # ratio: mask bins where MC == 0
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.where(mc_h2d > 0, data_h2d / mc_h2d, np.nan)
    ratio_norm = mcolors.Normalize(vmin=0, vmax=2)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=150)

    m0 = plot_2d(axes[0], data_h2d, x_edges, y_edges, "Data", norm=norm, cmap="viridis")
    m1 = plot_2d(axes[1], mc_h2d,   x_edges, y_edges, mc_label, norm=norm, cmap="viridis")
    m2 = plot_2d(axes[2], ratio,     x_edges, y_edges, "Data / MC", norm=ratio_norm, cmap="RdBu_r")

    fig.colorbar(m0, ax=axes[0], label="Count")
    fig.colorbar(m1, ax=axes[1], label="Count")
    fig.colorbar(m2, ax=axes[2], label="Ratio")

    for ax in axes:
        if plot["log_x"]:
            ax.set_xscale("log")
        if plot["log_y"]:
            ax.set_yscale("log")
        ax.set_xlabel(plot["x_label"])
        ax.set_ylabel(plot["y_label"])

    fig.suptitle(plot["title"], y=1.02)
    fig.tight_layout()

    subfolder = os.path.join(save_path, scan_name)
    os.makedirs(subfolder, exist_ok=True)
    plt.savefig(os.path.join(subfolder, f"{det_config}_2D.png"), bbox_inches="tight")
    plt.show()